---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-43: Creating and Deploying your own MCP Servers</h1>

# Learning agenda of this notebook
1. MCP Life Cycle
2. Design and Build a Local MCP Server
3. Run the MCP Server Locally in Debugging/Developer Mode and test it via MCP Inspector
4. Run the MCP Server Locally in Production Mode and Access it in Claude Desktop

# https://machinelearningmastery.com/building-a-simple-mcp-server-in-python/

<font color=purple> <h1> 1. MCP Communication Life Cycle</h1>
<h3 align="center"><div class="alert alert-success" style="margin: 20px">The MCP lifecycles describes the complete sequence of steps that govern how a MCP client and a MCP server establish communication, conduct their normal work, and then shut down properly.</div></h3>

> Visit: https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle
## (i) Initialization Phase
- **Initialization phase** is the starting handshake between the **client** and the **server**. Its goal is to make sure both sides agree on how to talk (protocol version) and what features (capabilities) are available.
    - **Client → Server: Initialization Request** The client sends an `initialize` request that includes Client info (name, protocol version) and Client capabilities (features it supports).
    - **Server → Client: Initialization Response** The server responds with its own details that includes Server info (name, protocol version) and Server capabilities (features it supports).
    - **Agreement on Protocol Version** Both sides compare protocol versions. They must use the same version (e.g., `2025-06-18`).
    - **Agreement on Capabilities** Client and server check what features both support. Only the common features are used in later communication.
    - **Client → Server: Initialized Notification** The client sends a final `initialized` notification. This means: *“I got your info, I’m ready to start normal operations.”*

```mermaid
sequenceDiagram
    participant Client as Client<br/>(Claude Desktop)
    participant Server as Server<br/>(MCP Calculator Server)
    
    rect rgb(200, 230, 255)
    Note over Client,Server: **1. Initialization Phase**
    Client->>Server: Contact server to start session
    Client->>Server: initialize(request)
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 1,<br/>"method": "initialize",<br/>"params": {<br/>  "protocolVersion": "2025-06-18",<br/>  "capabilities": {<br/>    "roots": {"listChanged": true},<br/>    "sampling": {}<br/>  },<br/>  "clientInfo": {<br/>    "name": "ClaudeDesktop",<br/>    "version": "2.5.1"<br/>  }<br/>}}
    Note right of Server: Negotiate protocol version<br/>Exchange capabilities<br/>(features, tools, resources)<br/>Share identity/metadata
    Server->>Client: initialize(response)<br/>(version, capabilities, metadata)
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 1,<br/>"result": {<br/>  "protocolVersion": "2025-06-18",<br/>  "capabilities": {<br/>    "logging": {},<br/>    "prompts": {"listChanged": true},<br/>    "resources": {<br/>      "subscribe": true,<br/>      "listChanged": true<br/>    },<br/>    "tools": {"listChanged": true}<br/>  },<br/>  "serverInfo": {<br/>    "name": "calculator",<br/>    "version": "1.0.0"<br/>  }<br/>}}
    Client->>Server: initialized(notification)
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"method": "notifications/initialized"<br/>}
    Note over Client,Server: Session established
    end
```

## (ii) Operation Phase
- **Operation** is the main working phase, where the client and server exchange real data and requests
    - Client requests → list tools, call tools, get resources, list prompts.
    - Server responses → executes tools, provide results, sends logs, updates.
    - Notifications  → handle dynamic changes like listChanged or subscribe.
    - Example: Claude client asks GitHub MCP server for repository file contents.

```mermaid
sequenceDiagram
    participant Client as Client<br/>(Claude Desktop)
    participant Server as Server<br/>(MCP Calculator Server)
     
    rect rgb(230, 255, 230)
    Note over Client,Server: **2. Operation Phase - Discover Tools**
    
    Client->>Server: tools/list
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 2,<br/>"method": "tools/list"<br/>}
    Server->>Client: tools/list response
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 2,<br/>"result": {<br/>  "tools": [<br/>    {<br/>      "name": "add",<br/>      "description": "Add two numbers",<br/>      "inputSchema": {<br/>        "type": "object",<br/>        "properties": {<br/>          "a": {"type": "number"},<br/>          "b": {"type": "number"}<br/>        },<br/>        "required": ["a", "b"]<br/>      }<br/>    },<br/>    {<br/>      "name": "subtract",<br/>      "description": "Subtract two numbers",<br/>      "inputSchema": {<br/>        "type": "object",<br/>        "properties": {<br/>          "a": {"type": "number"},<br/>          "b": {"type": "number"}<br/>        },<br/>        "required": ["a", "b"]<br/>      }<br/>    }<br/>  ]<br/>}}
    
    Note over Client,Server: **Use Tool - Addition**
    Client->>Server: tools/call (add)
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 3,<br/>"method": "tools/call",<br/>"params": {<br/>  "name": "add",<br/>  "arguments": {<br/>    "a": 15,<br/>    "b": 27<br/>  }<br/>}}
    Server->>Client: Result
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 3,<br/>"result": {<br/>  "content": [{<br/>    "type": "text",<br/>    "text": "Result: 42"<br/>  }],<br/>  "isError": false<br/>}}
    end
    
    rect rgb(255, 245, 230)
    Note over Client,Server: **Use Resources**
    
    Client->>Server: resources/list
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 4,<br/>"method": "resources/list"<br/>}
    Server->>Client: resources/list response
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 4,<br/>"result": {<br/>  "resources": [<br/>    {<br/>      "uri": "calc://docs/anthropic-info",<br/>      "name": "Anthropic Organization Info",<br/>      "description": "Company information",<br/>      "mimeType": "text/plain"<br/>    }<br/>  ]<br/>}}
    
    Note over Client,Server: **Subscribe to Resource Updates**
    Client->>Server: resources/subscribe
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 5,<br/>"method": "resources/subscribe",<br/>"params": {<br/>  "uri": "calc://docs/anthropic-info"<br/>}}
    Server->>Client: subscribe confirmation
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 5,<br/>"result": {}<br/>}
    
    Client->>Server: resources/read
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 6,<br/>"method": "resources/read",<br/>"params": {<br/>  "uri": "calc://docs/anthropic-info"<br/>}}
    Server->>Client: Resource content
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 6,<br/>"result": {<br/>  "contents": [{<br/>    "uri": "calc://docs/anthropic-info",<br/>    "mimeType": "text/plain",<br/>    "text": "Anthropic is an AI safety<br/>company founded in 2021..."<br/>  }]<br/>}}
    
    Note over Client,Server: **Server Notifies Resource Update**
    Server->>Client: notifications/resources/updated
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"method": "notifications/resources/updated",<br/>"params": {<br/>  "uri": "calc://docs/anthropic-info"<br/>}}
    
    Note over Client,Server: **Unsubscribe from Resource**
    Client->>Server: resources/unsubscribe
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 7,<br/>"method": "resources/unsubscribe",<br/>"params": {<br/>  "uri": "calc://docs/anthropic-info"<br/>}}
    Server->>Client: unsubscribe confirmation
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 7,<br/>"result": {}<br/>}
    end
    
    rect rgb(245, 230, 255)
    Note over Client,Server: **Operation Phase - Prompts**
    
    Client->>Server: prompts/list
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 8,<br/>"method": "prompts/list"<br/>}
    Server->>Client: prompts/list response
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 8,<br/>"result": {<br/>  "prompts": [<br/>    {<br/>      "name": "math_tutor",<br/>      "description": "Help with math problems",<br/>      "arguments": [<br/>        {<br/>          "name": "problem",<br/>          "description": "Math problem",<br/>          "required": true<br/>        }<br/>      ]<br/>    }<br/>  ]<br/>}}
    
    Client->>Server: prompts/get
    Note left of Client: {<br/>"jsonrpc": "2.0",<br/>"id": 9,<br/>"method": "prompts/get",<br/>"params": {<br/>  "name": "math_tutor",<br/>  "arguments": {<br/>    "problem": "15 + 27"<br/>  }<br/>}}
    Server->>Client: Prompt template
    Note right of Server: {<br/>"jsonrpc": "2.0",<br/>"id": 9,<br/>"result": {<br/>  "messages": [<br/>    {<br/>      "role": "user",<br/>      "content": {<br/>        "type": "text",<br/>        "text": "Please solve: 15 + 27"<br/>      }<br/>    }<br/>  ]<br/>}}
    end 
```

## (iii) Shutdown Phase
- **Shutdown** phase is the final stage of the MCP client–server lifecycle, when the sessions ends, either normally or due to error/timeout.
    - Ensures all resources are freed.
    - Ongoing requests are either completed or canceled gracefully.
    - Both client and server confirm that no further communication will occur.
```mermaid
sequenceDiagram
    participant Client as Client<br/>(Claude Desktop)
    participant Server as Server<br/>(GitHub MCP Server)

    rect rgb(255, 230, 230)
    Note over Client,Server: **3. Shutdown Phase**
    Client->>Server: shutdown request
    Note left of Client: {"jsonrpc": "2.0","id": 99,"method": "shutdown"}
    Server->>Client: shutdown response
    Note right of Server: {"jsonrpc": "2.0","id": 99,"result": null}
    Client->>Server: exit notification
    Note left of Client: {"jsonrpc": "2.0","method": "notifications/exit"}
    Note over Client,Server: Connection closed
    end
```

<font color=purple> <h1> 2. Design and Build Your Own MCP Server</h1>

#### Option 1: Server Class (Low-Level MCP)
- Gives you full control over everything: registering tools, defining schemas, wiring logic, startup, and error handling.
- Best for custom or very specific use cases where every detail matters.
- But it’s verbose and prone to mistakes, because you have to write a lot of boilerplate code.
- Think of it like programming in C/C++: very powerful, but less forgiving.
#### Option 2: FastMCP Class: (Higher-level abstraction)
- Built on top of the raw server logic, it handles most of the setup automatically.
- Automatically registers tools, infers input/output types from your functions, sets up the server, and manages client interactions.
- You just write functions and decorate them with @mcp.tool.
- Supports almost all MCP features and is actively maintained by the community.
- Think of it like programming in Python: simpler, faster, and more user-friendly than low-level code.

## Step 1. Setting up the Environment
- **Install uv (inside notebook):** Like `pip`, which is the classic Python’s package installer, `uv` is a newer, faster Python package + environment manager (like pip, but modernized). It is recommended to install and use `uv` if you are working with MCP servers. You can install `uv` by giving the following command:
```python
pip install uv
```
- **Create a new Project Folder:** Make an empty folder (for example, on your desktop) to host your MCP server project using the following commands:
```python
!mkdir -p ./mcp_demo_server
%cd ./mcp_demo_server
```
- **Initialize `uv` project:** To initialize a new Python project under `uv` use the following command that will create a set of baseline files in your folder, such as: `main.py`, `README.md`, `pyproject.toml`. These files help define your project's dependencies and environment.
```python
!uv init .
```
- **Add fastmcp dependency:** Now to install `fastmcp` and its dependencies into your `uv` environment, run the following command that will create a lock file (`uv.lock`) inside pwd, which pins exact versions. Do check the versions of FastMCP, MCP, Python, Platform and FastMCP root path.
```python
!uv add fastmcp
!uv run fastmcp version
```

In [ ]:
!uv run fastmcp version

## Step 2. Write the MCP server code
- In Jupyter, you can either use %%writefile to create main.py or edit externally.

In [8]:
%%writefile ./mcp_demo_server/main.py

import random # Import the random module from Python's standard library, that provides functions to generate random numbers
import requests, json
import datetime
import os
from dotenv import load_dotenv
from fastmcp import FastMCP # Import the FastMCP class from the fastmcp package that is a higher-level abstraction (wrapper) for building MCP servers quickly and easily
from fastmcp.resources import TextResource # For text/plain and application/json content
from fastmcp.resources import FileResource # For reading from files



# Load environment variables (API keys, etc.) from a .env file
load_dotenv('../keys/.env', override=True)
# Read WeatherAPI key from environment
weather_api_key = os.getenv("WEATHER_API_KEY")

# Create the server instance, giving it a name/identifier
#mcp = FastMCP(name="Demo-server")
mcp = FastMCP(name="Demo-server")


# -------------------------------------------------------------------------------------------------
# Define tools: to mark a function as a tool, we use the decorator @mcp.tool(), that is provided by FastMCP.
# Instead of writing JSON manually, You define tools using decorators. 
# MCP automatically generates:
        # Tool name
        # Description (from docstring)
        # Input schema (from type hints)
        # Output schema
        # Metadata
# This becmes part of MCP’s tool registry, which the model can discover and invoke dynamically.
# --------------------------------------------------------------------------------------------------

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and returns their sum."""
    return a+b

@mcp.tool()
def roll_dice(n_dice: int = 1) -> list[int]:
    """Role n_dice 6-sided dice and return the result"""
    return [random.randint(1,6) for _ in range(n_dice)] #generates a random integer between 1 and 6 (inclusive).

@mcp.tool()
def get_datetime() -> str:
    """Return the current date and time as a string."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@mcp.tool()
def fahrenheit_to_celsius(temp_f: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius."""
    return (temp_f - 32) * 5.0 / 9.0

@mcp.tool()
def reverse_string(text: str) -> str:
    """Reverse the given text string."""
    return text[::-1]

@mcp.tool()
def count_words(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@mcp.tool()
def get_weather(city:str = "Lahore") -> dict:
    """Fetch current weather info for a given city using WeatherAPI.com"""
    if not weather_api_key:
        return {"error": "Weather API key not set in .env file"}
    try:
        url = f"http://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={city}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        # Return structured info
        return {
            "Country": data['location']['country'],
            "City": data['location']['name'],
            "Latitude": data['location']['lat'],
            "Longitude": data['location']['lon'],
            "Temperature (C)": data['current']['temp_c'],
            "Temperature (F)": data['current']['temp_f'],
            "Condition": data['current']['condition']['text'],
            "Humidity (%)": data['current']['humidity'],
            "Wind (km/h)": data['current']['wind_kph'],
            "Feels Like (C)": data['current']['feelslike_c'],
            "Last Updated": data['current']['last_updated'],
        }
    except Exception as e:
        return {"error": str(e)}

# ---------------------------------------------------------------------
# Define Resources (static or retrievable data for MCP clients)
# ---------------------------------------------------------------------

# Static Resource 1: A static greeting message
mcp.add_resource(
    TextResource(
        uri="local://greeting",          # A unique identifier or address for the resource.
        name="greeting",                 # A human-readable name for the resource.
        description="A simple greeting message",     # Short explanation of what the resource is.
        text="Hello from your local MCP server! 🎉"  # The actual content of the resource (the text that clients will get).
    )
)

# Static Resource 2: Sample JSON data
sample_json_data = {"students": ["Hadeed", "Yashal", "Zalaid", "Miraal"], "class": "AI Basics"}
mcp.add_resource(
    TextResource(
        uri="local://sample-data",
        name="sample-data",
        description="Sample student data",
        text=json.dumps(sample_json_data, indent=2)
    )
)



# ---------------------------------------------------------------------
# Define Prompts 
# ---------------------------------------------------------------------


if __name__ == "__main__":
    mcp.run()

Overwriting ./mcp_demo_server/main.py


In [5]:
!uv run fastmcp run ./mcp_demo_server/main.py



╭────────────────────────────────────────────────────────────────────────────╮
│                                                                            │
│        _ __ ___  _____           __  __  _____________    ____    ____     │
│       _ __ ___ .'____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __ \    │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / / /    │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/ /     │
│    _ __ ___ /_/    \____/____/\__/_/  /_/\____/_/      /_____(*)____/      │
│                                                                            │
│                                                                            │
│                                FastMCP  2.0                                │
│                                                                            │
│                                                                            │
│                 🖥️  Server name:     Demo-server

<font color=purple> <h1> 3. Run the MCP Server in Debugging/Developer Mode and test it via MCP Inspector</h1>
## a. What is MCP Inspector?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">MCP Inspector is a web-based development tool that can connect directly to your MCP server—or sit between Claude Desktop and the server—to capture and display all JSON-RPC messages in real time without interfering with communication.</div></h3>

<div style="flex: 1; text-align: center;">
    <img src="../images/mcp-inspector1.png" alt="MCP Inspector" style="max-width: 1500px; height: 800px;">
</div>

- **Server Connection Panel** allows you to configure and establish connections to MCP servers. It displays connection parameters (transport type, command, arguments, environment variables) and provides controls to start, stop, and manage server connections. This is where you specify which MCP server to connect to and monitor its connection status.
    - Transport Type: Defines how data moves between the client and the MCP server (stdio, SSE, Streamable HTTP)
    - Command: This is the executable or interpreter that  launches your MCP server process.
    - Arguments: This is the list of parameters passed to the command, when launching the server.
    - Environment Variables: These are key-value pairs defining the environment in which the MCP server runs. These variables tell the server, where to find dependencies or binaries. Inside MCP Inspector, you can add, remove environment variables like (HOME, PATH, SHELL, USER, WEATHER_API) by clicking the button.
    - Authentication: Describes how the client authenticates to the MCP server. Most local MCP servers donot require authentication, IF it is a remote or cloud MCP server, you might see tokens, API keys or OAuth configurations.
    - Configuration: This selection lists user supplied or server-defined configuration parameters, like Request time out, Request timeout on p, MAximum total timeout, etc.
- **Resources, Tools, Prompts Panel** is a tabbed interface that displays the capabilities exposed by the connected MCP server, allowing you to explore and interact with the server's functionality.
    - Resources: Lists available resources (URIs, content) that the server provides access to
    - Tools: Shows executable tools with their schemas, parameters, and descriptions that can be invoked
    - Prompts: Displays prompt templates available from the server with their arguments and usage
- **History Panel** maintains a chronological log of all interactions between the inspector and the MCP server. It shows request/response pairs, including tool invocations, resource reads, prompt executions, and their results. This provides a complete audit trail for debugging and understanding the communication flow. 
- **Server Notification Panel** displays server-initiated notifications and events. These are asynchronous messages sent by the server to inform about state changes, resource updates, progress notifications, or other events that don't require a direct response. It helps monitor the server's ongoing activity and lifecycle events.
- **Error Output Panel** displays MCP Server logs like error messages, warnings, and debugging information that helps troubleshoot issues with server communication or implementation problems. 


- https://modelcontextprotocol.io/docs/tools/inspector
- **MCP Inspector** - https://github.com/modelcontextprotocol/inspector
- Give this command: `!npx @modelcontextprotocol/inspector` to run MCP Inspector

### Installing and Running MCP Inspector 
- **Option 1:** Use `npm` to install it permanently, then run it in standalone mode (recommended if you use it often)
```python
npm install -g @modelcontextprotocol/inspector
mcp-inspector
```
- **Option 2:** Use `npx` that will download and run it temprrarily
```python
!npx @modelcontextprotocol/inspector
```

## b. Run MCP Server in Debug/Development Mode
- The following command will run the MCP server  specified in main.py and automatically launches the MCP Inspector (used for Development, debugging)
- The `dev` subcommand launches your MCP server `main.py` in a development environment (i.e., the server isn't fully registered or installed).
- It will automatically open the MCP Inspector in your browser (http://localhost:port#)
- Remember, to stop the server when you're done testing: Press I, I (interrupt kernel) in Jupyter Or use Kernel → Interrupt from the menu
- If on running the server again and again , you see an error of port already busy use following two commands to detect the PID of the already running server and kill it
```bash
!lsof -i :<port#>
!kill -9 <PID>
```

In [6]:
!uv run fastmcp dev ./mcp_demo_server/main.py

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼Starting MCP inspector...
⚙️ Proxy server listening on 127.0.0.1:6277
🔑 Session token: 3fb0d1b7ecacc4a92c24df3d6e27c95fc80c48a3973e4bbc0793e1dd9e4bc9e6
Use this token to authenticate requests or set DANGEROUSLY_OMIT_AUTH=true to disable auth

🔗 Open inspector with token pre-filled:
   http://localhost:6274/?MCP_PROXY_AUTH_TOKEN=3fb0d1b7ecacc4a92c24df3d6e27c95fc80c48a3973e4bbc0793e1dd9e4bc9e6

🔍 MCP Inspector is up and running at http://127.0.0.1:6274 🚀
New STDIO connection request
Query parameters: {"command":"fastmcp","args":"run ./mcp_demo_server/main.py --no-banner","env":"{\"HOME\":\"/Users/arif\",\"LOGNAME\":\"arif\",\"PATH\":\"/Users/arif/.npm/_npx/5a9d879542beca3a/node_modules/.bin:/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-Arif/Agentic AI Course/Notebooks/node_modules/.bin:/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-Arif/Agentic AI Course/node_modules/.bin:/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-

In [4]:
!lsof -i :6274

In [2]:
!kill -9 97428

<font color=purple> <h1> 4. Run the MCP Server in Production Mode and Access it in Claude Desktop</h1>
## a. Run server in production mode on your Local Machine
- You can run the server using the following command inside the Jupyter notebook or on a terminal using the following commands:
```bash
!uv run fastmcp run ./mcp_demo_server/main.py
$  /Users/arif/.local/bin/uv run fastmcp run "/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-Arif/Agentic AI Course/Notebooks/mcp_demo_server/main.py"
```
- Now all the clients running on your machine can connect to this local MCP server. This is how  you’d run it in production (not just testing/debugging).


## b. Accessing Demo Server from Claude Desktop via its Config File
- Now we want the Claude Desktop app to read the config file (e.g., claude_desktop_config.json) and:
    - Spawns the server using "command" + "args".
    - Connects to it via STDIO.
- Add appropriate configurations for the Demo-server inside the `claude_desktop_config.json` file. The complete snapshot of the configuration that we have done so far is given below:
```bash
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/arif/Downloads"
      ],
      "env": {},
      "transport": "stdio",
      "type": null,
      "cwd": null,
      "timeout": null,
      "description": null,
      "icon": null,
      "authentication": null
    },
    
    "github": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-github"
      ],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "github_pat_11....."
      },
      "transport": "stdio",
      "type": null,
      "cwd": null,
      "timeout": null,
      "description": null,
      "icon": null,
      "authentication": null
    },

"fetch": {
      "command": "/Users/arif/.local/bin/uvx",
      "args": [
        "mcp-server-fetch"
      ],
      "env": {},
      "transport": "stdio",
      "type": null,
      "cwd": null,
      "timeout": null,
      "description": null,
      "icon": null,
      "authentication": null
    },
    "manim-server": {
      "command": "/opt/anaconda3/envs/llms/bin/python",
      "args": [
        "/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-Arif/Agentic AI Course/Notebooks/manim-mcp-server/src/manim_server.py"
      ],
      "env": {
        "MANIM_EXECUTABLE": "/opt/anaconda3/envs/llms/bin/manim"
      },
      "transport": "stdio",
      "type": null,
      "cwd": null,
      "timeout": null,
      "description": null,
      "icon": null,
      "authentication": null
    },
    "google-drive": {
      "command": "npx",
      "args": ["-y", "@piotr-agier/google-drive-mcp"],
      "env": {
        "GOOGLE_DRIVE_OAUTH_CREDENTIALS": "Library/Application Support/Claude/claude_gdrive_secrets.json"
      }
    },
  
 "Demo-server": {
      "command": "/Users/arif/.local/bin/uv",
      "args": [
        "run",
        "--with",
        "fastmcp",
        "fastmcp",
        "run",
        "/Users/arif/Documents/00 AI Course/9-Generative AI/GenAI-with-Arif/Agentic AI Course/Notebooks/mcp_demo_server/main.py"
      ],
      "env": {"WEATHER_API_KEY": "your weather api key"},
      "transport": "stdio",
      "type": null,
      "cwd": null,
      "timeout": null,
      "description": null,
      "icon": null,
      "authentication": null
    }
  }
}

```